In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
from scripts.plotting import *
from scripts.denoising import *
from scripts.TPS import *

In [ ]:
X = np.array(pd.read_csv("./data/s_curve/uniform/s_curve_noisy_position_matrix.csv"))
Y = np.array(pd.read_csv("./data/s_curve/uniform/s_curve_noisy_velocity_matrix.csv"))
t = pd.read_csv("./data/s_curve/uniform/s_curve_gt_latent_time_vector.csv")
t = list(t["t"])

X_gt = np.array(pd.read_csv("./data/s_curve/uniform/s_curve_gt_position_matrix.csv"))
Y_gt = np.array(pd.read_csv("./data/s_curve/uniform/s_curve_gt_velocity_matrix.csv"))

# X = X_gt
# Y = Y_gt

X.shape, Y.shape

In [ ]:
# Visualize using the provided function
plot_3d_with_quiver(
    X,
    Y,
    t,
    arrow_size=0.2
)

In [ ]:
import umap

umap_reducer = umap.UMAP(n_neighbors=15, min_dist=0.5, n_components=2, random_state=42)
X_2d = umap_reducer.fit_transform(X)
plot_2d(X_2d, t, "UMAP")

In [ ]:
tps = ThinPlateSpline(X_2d, n_control_points=1000)
tps.fit(X, dof_target=50)
metrics = tps.evaluate_fit(X)
metrics

In [ ]:
X_smoothed = tps.predict(X_2d)
plot_3d(X_smoothed,t)

In [ ]:
tps = ThinPlateSpline(X_2d)
tps.fit(X, dof_target=30)

In [ ]:
X_smoothed = tps.predict(X_2d)
plot_3d(X_smoothed,t)

In [ ]:
projected_velocities = tps.project_velocities(X_2d, Y)
tps_vf = ThinPlateSpline(X_2d)
tps_vf.fit(projected_velocities, dof_target=30)
smoothed_velocities = tps_vf.predict(X_2d) 

plot_2d_quiver(X_2d, smoothed_velocities, t, scale=5, cmap='coolwarm', arrow_color='black')

In [ ]:
def plot_all(X_2d, smoothed_velocities, t, tps, quiver_scale=5):
    """
    Plots three subplots in a row:
    1. 2D scatter plot of X_2d
    2. 2D quiver plot with smoothed velocities
    3. 3D scatter plot of transformed X_smoothed
    
    Args:
        X_2d (np.ndarray): 2D points.
        smoothed_velocities (np.ndarray): 2D velocity vectors.
        t (array-like): Color mapping values.
        tps: Thin Plate Spline (TPS) model for transformation.
    """
    X_smoothed = tps.predict(X_2d)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor="white")
    
    # Plot 2D scatter
    axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=t, s=50, alpha=0.8)
    axes[0].set_title("2D Scatter Plot")
    axes[0].xaxis.set_major_locator(ticker.MultipleLocator(1))
    axes[0].yaxis.set_major_locator(ticker.MultipleLocator(1))
    
    # Plot 2D quiver
    axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=t, cmap='coolwarm', s=10, alpha=0.3)
    axes[1].quiver(X_2d[:, 0], X_2d[:, 1], smoothed_velocities[:, 0], smoothed_velocities[:, 1], 
                    color='black', angles='xy', scale_units='xy', scale=quiver_scale)
    axes[1].set_title("2D Quiver Plot")
    axes[1].xaxis.set_major_locator(ticker.MultipleLocator(1))
    axes[1].yaxis.set_major_locator(ticker.MultipleLocator(1))
    
    # Plot 3D scatter
    ax3d = fig.add_subplot(133, projection='3d')
    ax3d.scatter(X_smoothed[:, 0], X_smoothed[:, 1], X_smoothed[:, 2], c=t, s=50, alpha=0.8)
    ax3d.set_title("3D Scatter Plot")
    ax3d.view_init(azim=-60, elev=9)
    
    plt.show()

In [ ]:
import numpy as np
from scipy.optimize import minimize

# Global cache to store previously computed loss and gradient
_loss_cache = None
_grad_cache = None

def compute_loss_and_gradient(X_flat, tps, tps_vf, X, Y, lam, mu, k=10, sigma=1.0):
    """Computes both the function value and gradient, including graph Laplacian smoothness."""
    X_2d = X_flat.reshape((-1, 2))

    # Compute shared values
    X_smoothed = tps.predict(X_2d)
    jacobians_f = tps.compute_tps_jacobians(X_2d)
    hessian_f = tps.compute_tps_hessians(X_2d)

    # Loss L1 (Reconstruction loss)
    loss1 = np.sum((X - X_smoothed) ** 2)

    # Loss L2 (Velocity field constraint)
    vf_smoothed = tps_vf.predict(X_2d)
    Y_smoothed = np.einsum("naj, nj -> na", jacobians_f, vf_smoothed)
    loss2 = np.sum((Y - Y_smoothed) ** 2)
    
    # Loss L3 (Graph Laplacian smoothness)
    D_high = cdist(X, X, metric='sqeuclidean')  # High-dimensional pairwise distances
    neighbors = np.argsort(D_high, axis=1)[:, 1:k+1]  # k-NN indices

    W = np.exp(-D_high / (2 * sigma**2))  # Weight matrix
    row_idx = np.repeat(np.arange(X.shape[0]), k)
    col_idx = neighbors.ravel()
    W_values = W[row_idx, col_idx]  # Only for k-NN pairs

    X2D_diff = X_2d[row_idx] - X_2d[col_idx]  # Differences in 2D space
    loss3 = np.sum(W_values * np.sum(X2D_diff**2, axis=1))  # Scalar loss

    # Compute Gradients
    grad1 = -2 * np.einsum("nai,na -> ni", jacobians_f, (X - X_smoothed))

    A = -2 * (Y - Y_smoothed)
    B = np.einsum("naij, nj -> nai", hessian_f, vf_smoothed)
    C = np.einsum("naj, nji -> nai", jacobians_f, tps_vf.compute_tps_jacobians(X_2d))
    grad2 = np.einsum("na, nai -> ni", A, (B + C))
    
    # Compute Gradient of L3
    grad3 = np.zeros_like(X_2d)
    np.add.at(grad3, row_idx, 2 * W_values[:, None] * X2D_diff)
    np.add.at(grad3, col_idx, -2 * W_values[:, None] * X2D_diff)  # Symmetric update

    # Total Loss and Gradient
    total_loss_value = loss1 + lam * loss2 + mu * loss3
    total_grad_value = (grad1 + lam * grad2 + mu * grad3).flatten()

    return total_loss_value, total_grad_value

def total_loss(X_flat, tps, tps_vf, X, Y, lam, mu):
    """Wrapper for computing and caching the loss function."""
    global _loss_cache, _grad_cache
    
    # Check cache
    if _loss_cache is not None and np.allclose(X_flat, _loss_cache[0]):
        return _loss_cache[1]

    # Compute function and gradient together
    loss_value, grad_value = compute_loss_and_gradient(X_flat, tps, tps_vf, X, Y, lam, mu)

    # Cache results
    _loss_cache = (X_flat.copy(), loss_value)
    _grad_cache = (X_flat.copy(), grad_value)

    return loss_value

def total_gradient(X_flat, tps, tps_vf, X, Y, lam, mu):
    """Wrapper for computing and caching the gradient."""
    global _grad_cache

    # Check cache
    if _grad_cache is not None and np.allclose(X_flat, _grad_cache[0]):
        return _grad_cache[1]

    # Compute function and gradient together
    loss_value, grad_value = compute_loss_and_gradient(X_flat, tps, tps_vf, X, Y, lam, mu)

    # Cache results
    _loss_cache = (X_flat.copy(), loss_value)
    _grad_cache = (X_flat.copy(), grad_value)

    return grad_value


def optimize_total(tps, tps_vf, X_2d, X, Y, lam, mu, method="L-BFGS-B", tol=1e-4, disp=False):
    """
    Optimizes the total loss function using L-BFGS-B (or other methods).
    
    Args:
        tps: TPS model for X smoothing.
        tps_vf: TPS model for vector field.
        X_2d: Initial 2D points (n, 2).
        X: Original high-dimensional data points.
        Y: Vector field data at each point.
        lam: Regularization parameter.
        method: Optimization method (default: 'L-BFGS-B').
        tol: Tolerance for stopping criterion (higher means faster but less precise).
    
    Returns:
        X_optimized: Optimized 2D coordinates.
        result: SciPy optimization result object.
    """
    global _loss_cache, _grad_cache
    _loss_cache, _grad_cache = None, None  # Reset cache before optimization

    X_flat = X_2d.flatten()  # Flatten input for optimizer

    result = minimize(
        fun=total_loss,
        x0=X_flat,
        jac=total_gradient,
        args=(tps, tps_vf, X, Y, lam, mu),
        method=method,
        tol=tol,  # Set higher tolerance
        options={
            "gtol": tol,   # Gradient norm stopping criterion
            "ftol": tol,   # Function value stopping criterion
            "disp": disp   # Display optimization progress
        }
    )
    X_optimized = result.x.reshape((-1, 2))  # Reshape back to 2D
    return X_optimized, result


lam = 1
mu = 1
%time X_optimized, optimization_result = optimize_total(tps, tps_vf, X_2d, X, Y, lam, mu, disp=True)

In [ ]:
plot_2d(X_2d, t)
plot_2d(X_optimized, t)

In [ ]:
# Step 1: Initialize with PCA
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)  # Initial embedding
tps = ThinPlateSpline(X_2d)
tps.fit(X, dof_target=30)
projected_velocities = tps.project_velocities(X_2d, Y)
tps_vf = ThinPlateSpline(X_2d)
tps_vf.fit(projected_velocities, dof_target=30)
smoothed_velocities = tps_vf.predict(X_2d)

plot_all(X_2d, smoothed_velocities, t, tps)

lam = 1
mu = 1
lam_functional = tps.lambda_reg
num_iterations = 3
for i in range(num_iterations):
    print(f"Iteration {i+1}")

    # Step 2: Fit the thin-plate spline (TPS)
    tps = ThinPlateSpline(X_2d)
    tps.fit(X, lambda_reg=lam_functional)
    
    projected_velocities = tps.project_velocities(X_2d, Y)
    tps_vf = ThinPlateSpline(X_2d)
    tps_vf.fit(projected_velocities, lambda_reg=lam_functional)
    
    X_2d, optimization_result = optimize_total(tps, tps_vf, X_2d, X, Y, lam, mu)

    vf_smoothed = tps_vf.predict(X_2d) 
    plot_all(X_2d, vf_smoothed, t, tps)

In [ ]:
# Step 1: Initialize with umap
umap_reducer = umap.UMAP(n_neighbors=15, min_dist=0.5, n_components=2, random_state=42)
X_2d = umap_reducer.fit_transform(X)

tps = ThinPlateSpline(X_2d)
tps.fit(X, dof_target=30)
projected_velocities = tps.project_velocities(X_2d, Y)
tps_vf = ThinPlateSpline(X_2d)
tps_vf.fit(projected_velocities, dof_target=30)
smoothed_velocities = tps_vf.predict(X_2d)

plot_all(X_2d, smoothed_velocities, t, tps)

lam = 1
mu = 1
lam_functional = tps.lambda_reg
num_iterations = 3
for i in range(num_iterations):
    print(f"Iteration {i+1}")

    # Step 2: Fit the thin-plate spline (TPS)
    tps = ThinPlateSpline(X_2d)
    tps.fit(X, lambda_reg=lam_functional)
    
    projected_velocities = tps.project_velocities(X_2d, Y)
    tps_vf = ThinPlateSpline(X_2d)
    tps_vf.fit(projected_velocities, lambda_reg=lam_functional)
    
    X_2d, optimization_result = optimize_total(tps, tps_vf, X_2d, X, Y, lam, mu)

    vf_smoothed = tps_vf.predict(X_2d) 
    plot_all(X_2d, vf_smoothed, t, tps)

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42)
X_2d = tsne.fit_transform(X)

tps = ThinPlateSpline(X_2d)
tps.fit(X, dof_target=50)
projected_velocities = tps.project_velocities(X_2d, Y)
tps_vf = ThinPlateSpline(X_2d)
tps_vf.fit(projected_velocities, dof_target=50)
smoothed_velocities = tps_vf.predict(X_2d)

plot_all(X_2d, smoothed_velocities, t, tps)

lam = 1
lam_functional = tps.lambda_reg
num_iterations = 3
for i in range(num_iterations):
    print(f"Iteration {i+1}")

    # Step 2: Fit the thin-plate spline (TPS)
    tps = ThinPlateSpline(X_2d)
    tps.fit(X, lambda_reg=lam_functional)
    
    projected_velocities = tps.project_velocities(X_2d, Y)
    tps_vf = ThinPlateSpline(X_2d)
    tps_vf.fit(projected_velocities, lambda_reg=lam_functional)
    
    X_2d, optimization_result = optimize_total(tps, tps_vf, X_2d, X, Y, lam, mu)

    vf_smoothed = tps_vf.predict(X_2d) 
    plot_all(X_2d, vf_smoothed, t, tps)

In [ ]:
from sklearn.manifold import SpectralEmbedding

spectral = SpectralEmbedding(n_components=2, n_neighbors=15, random_state=42)
X_2d = spectral.fit_transform(X)

tps = ThinPlateSpline(X_2d)
tps.fit(X, dof_target=12)
projected_velocities = tps.project_velocities(X_2d, Y)
tps_vf = ThinPlateSpline(X_2d)
tps_vf.fit(projected_velocities, dof_target=12)
smoothed_velocities = tps_vf.predict(X_2d)

plot_all(X_2d, smoothed_velocities, t, tps, quiver_scale=15)

lam = 1
lam_functional = tps.lambda_reg
num_iterations = 3
for i in range(num_iterations):
    print(f"Iteration {i+1}")

    # Step 2: Fit the thin-plate spline (TPS)
    tps = ThinPlateSpline(X_2d)
    tps.fit(X, lambda_reg=lam_functional)
    
    projected_velocities = tps.project_velocities(X_2d, Y)
    tps_vf = ThinPlateSpline(X_2d)
    tps_vf.fit(projected_velocities, lambda_reg=lam_functional)
    
    X_2d, optimization_result = optimize_total(tps, tps_vf, X_2d, X, Y, lam, mu)

    vf_smoothed = tps_vf.predict(X_2d) 
    plot_all(X_2d, vf_smoothed, t, tps, quiver_scale=15)

In [ ]:
from sklearn.manifold import Isomap

isomap = Isomap(n_components=2, n_neighbors=10)
X_2d = isomap.fit_transform(X)

tps = ThinPlateSpline(X_2d)
tps.fit(X, dof_target=20)
projected_velocities = tps.project_velocities(X_2d, Y)
tps_vf = ThinPlateSpline(X_2d)
tps_vf.fit(projected_velocities, dof_target=20)
smoothed_velocities = tps_vf.predict(X_2d)

plot_all(X_2d, smoothed_velocities, t, tps)

lam = 1
lam_functional = tps.lambda_reg
num_iterations = 3
for i in range(num_iterations):
    print(f"Iteration {i+1}")

    # Step 2: Fit the thin-plate spline (TPS)
    tps = ThinPlateSpline(X_2d)
    tps.fit(X, lambda_reg=lam_functional)
    
    projected_velocities = tps.project_velocities(X_2d, Y)
    tps_vf = ThinPlateSpline(X_2d)
    tps_vf.fit(projected_velocities, lambda_reg=lam_functional)
    
    X_2d, optimization_result = optimize_total(tps, tps_vf, X_2d, X, Y, lam, mu)

    vf_smoothed = tps_vf.predict(X_2d) 
    plot_all(X_2d, vf_smoothed, t, tps)

In [ ]:
from sklearn.manifold import Isomap


# Example usage:
np.random.seed(42)
X_2d = np.random.uniform(0, 1, size=(1000, 2))

tps = ThinPlateSpline(X_2d)
tps.fit(X, dof_target=20)
projected_velocities = tps.project_velocities(X_2d, Y)
tps_vf = ThinPlateSpline(X_2d)
tps_vf.fit(projected_velocities, dof_target=20)
smoothed_velocities = tps_vf.predict(X_2d)

plot_all(X_2d, smoothed_velocities, t, tps)

lam = 1
lam_functional = tps.lambda_reg
num_iterations = 10
for i in range(num_iterations):
    print(f"Iteration {i+1}")

    # Step 2: Fit the thin-plate spline (TPS)
    tps = ThinPlateSpline(X_2d)
    tps.fit(X, lambda_reg=lam_functional)
    
    projected_velocities = tps.project_velocities(X_2d, Y)
    tps_vf = ThinPlateSpline(X_2d)
    tps_vf.fit(projected_velocities, lambda_reg=lam_functional)
    
    X_2d, optimization_result = optimize_total(tps, tps_vf, X_2d, X, Y, lam, mu)

    vf_smoothed = tps_vf.predict(X_2d) 
    plot_all(X_2d, vf_smoothed, t, tps)